# 시간표 수집

평일·토요일·일요일/공휴일을 나누어 수집하고, 검증한 시간표를 경로 탐색에서 사용합니다. 다시 실행하면 저장한 역은 재사용합니다. 학습이나 경로 검색은 실행하지 않습니다.


In [ ]:
from pathlib import Path
import sys

candidates = [Path.cwd(), Path.cwd().parent, Path.cwd() / 'bicycle']
ROOT = next((p.resolve() for p in candidates if (p / 'src/map_schedule_collection.py').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('프로젝트의 notebooks 폴더에서 열어 주세요.')
sys.path.insert(0, str(ROOT / 'src'))
from map_schedule_collection import collect
from collected_rail import compile_collected
FOLDER = ROOT / 'data/transit/map_timetables'
print('수집 위치:', FOLDER)


## 수집

수도권 지하철을 역 단위로 저장합니다. 기본 요청 간격은 1.5초이며, 한 번에 최대 1,000회를 요청합니다. 실패해도 저장된 응답은 유지됩니다. 7일 이내의 캐시는 재사용합니다.


In [ ]:
result = collect(FOLDER, max_requests=1000, interval=1.5)
print('저장:', len(result['stations']), '/', result.get('expected_stations'))
print('완료:', result.get('complete', False))
print('이번 요청:', result['requests_this_run'])
if not result.get('complete'):
    print('중단 사유:', result.get('stopped'))
    print('다시 Run All 하면 저장 지점부터 이어집니다.')


## 검증과 반영

수집이 완료된 경우에만 노선·요일별 시간표로 변환합니다. 불완전한 수집본은 사용 중인 시간표를 대체하지 않습니다. 실행 중인 웹 서버에는 재시작 후 반영됩니다.


In [ ]:
if result.get('complete'):
    manifest = compile_collected(FOLDER.parent)
    import pandas as pd
    display(pd.DataFrame.from_dict(manifest['rows_by_profile'], orient='index', columns=['역별 시각 기록 수']))
    print('수집 완료 노선:', ', '.join(manifest['complete_lines']))
    print('경로 탐색에 반영하는 노선:', ', '.join(manifest['active_lines']))
    print('사용 버전:', manifest['signature'])
else:
    print('기존 사용 시간표를 유지합니다.')


## 범위

토요일과 일요일은 분리하며 평일 공휴일도 휴일 표를 씁니다. 자정 이후 막차는 이전 운행일에 연결합니다. 열차번호·방향·지선·종착지를 확인해 연결하고 중간 역이 없으면 건너뛰지 않습니다.

역별 공개 시간표에 종착역 도착편이 없는 경우는 임의로 채우지 않습니다. 버스에는 아직 배차간격 기반 복원 구간이 남습니다. [수집 설명](../docs/TIMETABLE_COLLECTION.md)을 확인하세요.

중간역에서 운행을 끝내는 열차도 행선지로 구분합니다. 같은 방향·요일의 한 구간에서 여러 열차의 관측 소요시간이 모두 같을 때만 마지막 도착시각을 추정하고, 추정 여부를 별도로 저장합니다. 소요시간이 달라지거나 관측이 없으면 채우지 않습니다. 에버라인 주말의 열차번호도 임의로 만들지 않습니다.
